# ReplayGate — Feature Playground

**Conversation-level regression testing for multi-turn AI agents.**

Record a real agent conversation once — every LLM call, tool call, and session transition — into a
deterministic fixture, then replay it offline against a new model/prompt/version and assert
**cross-turn invariants**.

This notebook walks every shipped feature, end to end, **fully offline** (zero network):

1. The trace contract (`Conversation` / `Turn` / `Message` / `ToolCall` + cross-turn query helpers)
2. Built-in scenarios, reference agents & the invariant registry
3. Record a scenario offline (scripted LLM)
4. Inspect the fixture directory
5. Replay-and-diff
6. The cross-turn invariant suite (`replaygate.invariants`) — re-ask / forgotten constraint
7. The flagship confirmation invariant (`booked_only_after_confirmation`)
8. The `regress` gate (`run_regress` / `RegressReport` + `replaygate regress`)
9. Request keying (content-addressed recording)
10. Timing spans in DuckDB (OpenTelemetry GenAI)
11. Live providers (the recording seam) + how to go live
12. Build your own scenario + agent

> Run top-to-bottom. Each cell builds on the previous one.

## Setup

This notebook needs the project installed (`uv pip install -e .` or `pip install -e .`).
If you launched Jupyter from outside the project venv, select the project kernel, or run
`uv run jupyter lab` from the repo root.

In [ ]:
import tempfile
from pathlib import Path
from datetime import datetime, timezone

import replaygate

print("replaygate version:", replaygate.__version__)

# A scratch directory for the fixtures we record in this session.
WORK = Path(tempfile.mkdtemp(prefix="replaygate-playground-"))
print("scratch dir:", WORK)

## 1. The trace contract

Everything hangs off a small tree of Pydantic models. A `Conversation` carries `Turn`s; each `Turn`
has user/assistant `Message`s and the `ToolCall`s made that turn. The `Conversation` also exposes the
**query helpers** the regression detector needs — these are what make assertions *cross-turn* rather
than per-reply.

In [ ]:
from replaygate.trace.models import (
    Conversation, Turn, Message, ToolCall, SessionMeta, AgentStep,
)

now = datetime.now(timezone.utc)

conv = Conversation(
    id="demo-1",
    scenario="hand_built",
    channel="direct",
    session_meta=SessionMeta(session_id="s1", started_at=now),
    turns=[
        Turn(
            index=0,
            user_messages=[Message(role="user", content="what slots on 2026-07-01?", ts=now)],
            assistant_messages=[Message(role="assistant", content="10am and 3pm.", ts=now)],
            tool_calls=[ToolCall(name="search_slots", arguments={"date": "2026-07-01"},
                                 result={"slots": ["10am", "3pm"]}, call_id="c1")],
        ),
        Turn(
            index=1,
            user_messages=[Message(role="user", content="yes please, book 3pm", ts=now)],
            assistant_messages=[Message(role="assistant", content="Booked 3pm!", ts=now)],
            tool_calls=[ToolCall(name="book_appointment", arguments={"slot": "3pm"},
                                 result={"booked": True}, call_id="c2")],
        ),
    ],
    agent_version="dev",
    model="claude-haiku-4-5",
)

print("turns:", len(conv.turns))
print("all tool calls:", [tc.name for tc in conv.all_tool_calls()])
print("tool calls with a result:", [tc.name for tc in conv.tool_results()])

# Cross-turn query helper: was there an affirmation in any turn BEFORE this index?
print("user_confirmed_before(1):", conv.user_confirmed_before(1))  # only turn 0 ('what slots') → False
print("user_confirmed_before(2):", conv.user_confirmed_before(2))  # includes turn 1 ('yes') → True

## 2. Built-in scenarios, reference agents & the invariant registry

ReplayGate ships reference agents, each pinned to a **cross-turn invariant**, plus a clean (`*_happy`)
and a deliberately-broken (`*_regression`) variant that trips it. Each `ExampleSpec` bundles
everything the recorder needs to reproduce the run offline: the scenario, an agent factory, its tool
registry, and a **scripted LLM script** (so recording never touches a network).

The `replaygate.invariants` module maps every scenario to the invariants that must hold for it, via
`INVARIANTS_BY_SCENARIO` (queried with `invariants_for(name)`). That registry is what the `regress`
gate (section 8) runs after a replay.

In [ ]:
from replaygate.examples.scenarios import EXAMPLES
from replaygate.invariants import invariants_for

for name, spec in EXAMPLES.items():
    turns = " | ".join(" + ".join(t) for t in spec.scenario.user_turns)
    invs = [fn.__name__ for fn in invariants_for(name)]
    tag = "(regression)" if name.endswith("regression") else "(happy)"
    print(f"- {name:38s} {tag}  model={spec.model}")
    print(f"    invariants: {invs}")
    print(f"    user turns: {turns}")

## 3. Record a scenario offline (scripted LLM, zero network)

`record_conversation` drives the agent through the scenario's user turns. A `RecordingLLMClient` and
`ToolRecorder` sit in front of the agent's LLM/tools and log every call, keyed by content. Because we
feed it the **scripted** LLM (`scripted_llm_for`), this runs with no API key and no network.

The output is a `Fixture` — written to disk as a *directory*, not a blob.

In [ ]:
from replaygate.capture.record import record_conversation
from replaygate.capture.adapters import DirectAdapter
from replaygate.examples.scenarios import scripted_llm_for
from replaygate.store.fixtures import write_fixture

def record_builtin(name: str, out_dir: Path) -> Path:
    spec = EXAMPLES[name]
    fixture = record_conversation(
        agent_factory=spec.build_agent,
        inner_llm=scripted_llm_for(name),
        scenario=spec.scenario,
        adapter=DirectAdapter(),
        tools=spec.tools(),
        agent_version="dev",
        model=spec.model,
        recorded_at=datetime.now(timezone.utc),
    )
    write_fixture(str(out_dir), fixture)
    return out_dir

booking_dir = record_builtin("booking_happy", WORK / "booking_happy")
print("recorded → ", booking_dir)
for f in sorted(booking_dir.iterdir()):
    print(f"  {f.name:22s} {f.stat().st_size:>5d} bytes")

## 4. Inspect the fixture

A fixture directory is **readable JSON about what the agent actually did** — the whole point of
recording at the application seam rather than capturing HTTP bytes:

- `conversation.json` — the trace contract, serialized
- `llm_recording.json` — every LLM exchange, keyed by a sha256 over `(model, system, messages, tools)`
- `tool_recording.json` — every tool call + its result
- `spans.jsonl` — OpenTelemetry-aligned timing spans
- `meta.json` — scenario / agent version / model / recorded_at

In [ ]:
from replaygate.store.fixtures import read_fixture

fx = read_fixture(str(booking_dir))

print("== meta ==")
print(fx.meta.model_dump_json(indent=2))

print("\n== conversation turns ==")
for t in fx.conversation.turns:
    u = " ".join(m.content for m in t.user_messages)
    a = " ".join(m.content for m in t.assistant_messages)
    print(f"  turn {t.index}: user={u!r}")
    print(f"           asst={a!r}  tools={[tc.name for tc in t.tool_calls]}")

print("\n== llm_recording (content-addressed) ==")
for entry in fx.llm_recording:
    print(f"  key={entry['request_key'][:12]}…  -> {entry['response']['text']!r}")

print("\n== tool_recording ==")
for entry in fx.tool_recording:
    print(f"  {entry['tool']}({entry['args']}) -> {entry['result']}")

## 5. Replay-and-diff (zero network)

Replay re-runs the agent against the recorded LLM/tool exchanges (served by content key) — no
provider SDK is imported and no key is needed. `diff_conversations` then compares the recorded vs
replayed turns. An **empty diff means a faithful replay**.

In [ ]:
from replaygate.capture.replay import replay_conversation, diff_conversations

spec = EXAMPLES["booking_happy"]
replayed = replay_conversation(fx, spec.build_agent, spec.tools())
diffs = diff_conversations(fx.conversation, replayed)

print("diffs:", diffs)
print("OK — faithful replay" if not diffs else "MISMATCH")
print(f"{len(replayed.turns)} turns reproduced offline, zero network")

## 6. The cross-turn invariant suite

This is the class of bug ReplayGate exists for — defects that only exist *between* turns and pass
every per-turn assertion.

`replaygate.invariants` ships these as pure functions `Conversation -> InvariantResult`, where
`InvariantResult` carries `name`, `passed` (**True = the invariant held**), and a human `detail`. We
record a **clean** agent and its **broken** twin, then run the real invariant over each recorded
conversation — no hand-rolled detector needed.

### 6a. Support agent — *don't re-ask for an order id given earlier* (`order_id_never_reasked`)

In [ ]:
from replaygate.invariants import order_id_never_reasked

good = read_fixture(str(record_builtin("support_happy", WORK / "support_happy")))
bad = read_fixture(str(record_builtin("support_reask_regression", WORK / "support_reask")))

for label, f in [("support_happy", good), ("support_reask_regression", bad)]:
    turn1 = f.conversation.turns[1]
    asst = " ".join(m.content for m in turn1.assistant_messages)
    result = order_id_never_reasked(f.conversation)
    print(f"{label:28s} turn-1 reply: {asst!r}")
    print(f"{'':28s} {'HELD ' if result.passed else 'VIOLATED'} — {result.detail}")

### 6b. Profile agent — *honor a dietary constraint set earlier* (`dietary_constraint_honored`)

Same shape: the clean agent recommends a dairy-free dish (constraint carried forward); the broken
twin forgets and recommends pizza. The tell is in the recorded **tool-call arguments and result**,
which the invariant inspects.

In [ ]:
from replaygate.invariants import dietary_constraint_honored

good = read_fixture(str(record_builtin("profile_happy", WORK / "profile_happy")))
bad = read_fixture(str(record_builtin("profile_forgets_regression", WORK / "profile_forgets")))

for label, f in [("profile_happy", good), ("profile_forgets_regression", bad)]:
    recs = [tc for tc in f.conversation.all_tool_calls() if tc.name == "recommend_dish"]
    dish = recs[0].result if recs else None
    result = dietary_constraint_honored(f.conversation)
    print(f"{label:30s} recommended: {dish}")
    print(f"{'':30s} {'HELD ' if result.passed else 'VIOLATED'} — {result.detail}")

## 7. The flagship confirmation invariant (`booked_only_after_confirmation`)

The booking invariant: *never book before the user confirmed*. It builds on
`Conversation.user_confirmed_before`, which detects an affirmation (`yes`, `sure`, `confirm`, …) in
turns up to a given index — now **punctuation-tolerant**, so `"yes, book 3pm"` counts. The invariant
flags any turn that calls `book_appointment` without a confirmation on or before it.

The clean `booking_happy` fixture holds; the shipped `booking_books_without_confirm_regression`
scenario — a `BookingAgent` that books on a turn where the user only said *"hmm, let me think about
it"* — trips it.

In [ ]:
from replaygate.invariants import booked_only_after_confirmation

# Clean fixture (recorded in section 3) holds the invariant.
booking = read_fixture(str(booking_dir)).conversation
happy = booked_only_after_confirmation(booking)
print(f"booking_happy                          {'HELD ' if happy.passed else 'VIOLATED'} — {happy.detail}")

# Shipped regression scenario: books after 'hmm, let me think about it' (no confirmation).
bad_dir = record_builtin("booking_books_without_confirm_regression", WORK / "booking_bad")
bad_booking = read_fixture(str(bad_dir)).conversation
bad = booked_only_after_confirmation(bad_booking)
print(f"booking_books_without_confirm_regression {'HELD ' if bad.passed else 'VIOLATED'} — {bad.detail}")

## 9. Request keying (content-addressed recording)

Recording matches on **meaning, not byte order**. `request_key` is a sha256 over
`(model, system, messages, tools)` with sorted keys, so semantically-identical requests collide
regardless of dict ordering. That key is how replay finds the right recorded response.

In [ ]:
import subprocess

from replaygate.regress import run_regress, RegressReport

def regress_dir(fixture_dir):
    fx = read_fixture(str(fixture_dir))
    spec = EXAMPLES[fx.conversation.scenario]
    return run_regress(fx, spec.build_agent, spec.tools())   # replays offline, runs invariants_for(scenario)

print("== run_regress (programmatic) ==")
for fixture_dir in [booking_dir, bad_dir]:
    report: RegressReport = regress_dir(fixture_dir)
    print(f"[{'OK  ' if report.passed else 'FAIL'}] {report.scenario}")
    for r in report.results:
        print(f"         [{'PASS' if r.passed else 'FAIL'}] {r.name}: {r.detail}")

# The same gate through the CLI — exit 0 = all invariants held, 1 = violation, 2 = bad/unknown fixture.
print("\n== replaygate regress (CLI, the CI gate) ==")
for fixture_dir in [booking_dir, bad_dir]:
    proc = subprocess.run(["replaygate", "regress", str(fixture_dir)], capture_output=True, text=True)
    print(f"$ replaygate regress {fixture_dir.name}  → exit {proc.returncode}")
    print("  " + proc.stdout.strip().replace("\n", "\n  "))

## 10. Timing spans in DuckDB (OpenTelemetry GenAI)

Timing spans are stored in DuckDB with attributes aligned to OpenTelemetry's **GenAI semantic
conventions** (`gen_ai.request.model`, `gen_ai.agent.name`, …). Here we use an in-memory DuckDB so
nothing is written to disk.

In [ ]:
from replaygate.capture.llm import request_key, RecordingLLMClient, LLMResponse

a = request_key("m", "sys", [{"role": "user", "content": "hi"}], [])
# Same content, keys built in a different order → identical hash.
b = request_key("m", "sys", [{"content": "hi", "role": "user"}], [])
print("deterministic & order-independent:", a == b, a[:16], "…")

# Round-trip: record against a tiny fake inner client, then replay from the log.
class FakeInner:
    def create(self, model, system, messages, tools):
        return LLMResponse(text=f"echo: {messages[-1]['content']}")

log = []
rec = RecordingLLMClient(FakeInner(), mode="record", recording=log)
resp = rec.create("m", "sys", [{"role": "user", "content": "ping"}], [])
print("recorded response:", resp.text, "| log entries:", len(log))

rep = RecordingLLMClient(inner=None, mode="replay", recording=log)  # inner=None → never calls out
print("replayed response:", rep.create("m", "sys", [{"role": "user", "content": "ping"}], []).text)

try:
    rep.create("m", "sys", [{"role": "user", "content": "unseen"}], [])
except KeyError as e:
    print("unseen request raises:", e)

## 11. Live providers (the recording seam)

To record against a **real** model, you swap the scripted LLM for a provider client. Each client
satisfies the same one-method `LLMClient` protocol, so the recorder slots in front unchanged —
Anthropic via the official SDK; OpenAI / OpenRouter / Ollama / Gemini via the OpenAI-compatible SDK.

```python
from replaygate.capture.providers import make_client
inner = make_client("anthropic", model="claude-haiku-4-5")   # reads ANTHROPIC_API_KEY
# ...then pass inner_llm=inner to record_conversation(...)
```

Or from the CLI:

```bash
replaygate record-live booking_happy ./fx --provider anthropic
replaygate record-live booking_happy ./fx --provider openai --model gpt-4o-mini
```

Below we demonstrate the **seam itself offline**: by injecting a fake SDK client (`client=…`) we can
see each provider translate the neutral tool schema into its own shape and parse the response back
into a uniform `LLMResponse` — no key, no network.

In [ ]:
from replaygate.store.spans import SpanStore
from replaygate.trace.models import SpanRecord

store = SpanStore(":memory:")
store.write([
    SpanRecord(trace_id="t1", span_id="s1", parent_id=None, operation="conversation",
               attributes={"gen_ai.agent.name": "booking", "scenario": "booking_happy"},
               start_ns=0, end_ns=5_000_000),
    SpanRecord(trace_id="t1", span_id="s2", parent_id="s1", operation="llm.create",
               attributes={"gen_ai.request.model": "claude-haiku-4-5", "gen_ai.operation.name": "chat"},
               start_ns=1_000_000, end_ns=3_000_000),
    SpanRecord(trace_id="t1", span_id="s3", parent_id="s1", operation="tool.call",
               attributes={"tool.name": "search_slots"}, start_ns=3_100_000, end_ns=3_400_000),
])

for s in store.read("t1"):
    dur_ms = (s.end_ns - s.start_ns) / 1e6
    print(f"  {s.operation:14s} {dur_ms:6.2f} ms  parent={s.parent_id}  attrs={s.attributes}")

## 10. Live providers (the recording seam)

To record against a **real** model, you swap the scripted LLM for a provider client. Each client
satisfies the same one-method `LLMClient` protocol, so the recorder slots in front unchanged —
Anthropic via the official SDK; OpenAI / OpenRouter / Ollama / Gemini via the OpenAI-compatible SDK.

```python
from replaygate.capture.providers import make_client
inner = make_client("anthropic", model="claude-haiku-4-5")   # reads ANTHROPIC_API_KEY
# ...then pass inner_llm=inner to record_conversation(...)
```

Or from the CLI:

```bash
replaygate record-live booking_happy ./fx --provider anthropic
replaygate record-live booking_happy ./fx --provider openai --model gpt-4o-mini
```

Below we demonstrate the **seam itself offline**: by injecting a fake SDK client (`client=…`) we can
see each provider translate the neutral tool schema into its own shape and parse the response back
into a uniform `LLMResponse` — no key, no network.

In [ ]:
import types
from replaygate.capture.providers import make_client, AnthropicClient, OpenAIClient
from replaygate.examples.booking_agent import booking_tool_schemas

print("registered providers:")
for p in ["anthropic", "openai", "openrouter", "ollama", "gemini"]:
    print("  -", p, "->", type(make_client(p, model="x", client=object())).__name__
          if p in {"openrouter", "ollama", "gemini"} else p)

tools = booking_tool_schemas()

# --- Anthropic seam (fake SDK) ---
anthropic_capture = {}
def anthropic_create(**kwargs):
    anthropic_capture.update(kwargs)
    return types.SimpleNamespace(
        id="msg_1", stop_reason="tool_use", model="claude-haiku-4-5",
        content=[
            types.SimpleNamespace(type="text", text="There are 10am and 3pm."),
            types.SimpleNamespace(type="tool_use", name="search_slots", input={"date": "2026-07-01"}),
        ],
    )
fake_anthropic = types.SimpleNamespace(messages=types.SimpleNamespace(create=anthropic_create))
aclient = AnthropicClient(client=fake_anthropic)
aresp = aclient.create("claude-haiku-4-5", "SYS", [{"role": "user", "content": "slots?"}], tools)
print("\n[anthropic] tool sent as:", anthropic_capture["tools"][0])
print("[anthropic] parsed LLMResponse:", aresp.text, "| tool_calls:", aresp.tool_calls)

# --- OpenAI-compatible seam (fake SDK) ---
openai_capture = {}
def openai_create(**kwargs):
    openai_capture.update(kwargs)
    msg = types.SimpleNamespace(content="There are 10am and 3pm.", tool_calls=[])
    return types.SimpleNamespace(id="cmpl_1", model="gpt-4o-mini",
                                 choices=[types.SimpleNamespace(message=msg, finish_reason="stop")])
fake_openai = types.SimpleNamespace(
    chat=types.SimpleNamespace(completions=types.SimpleNamespace(create=openai_create)))
oclient = OpenAIClient(client=fake_openai)
oresp = oclient.create("gpt-4o-mini", "SYS", [{"role": "user", "content": "slots?"}], tools)
print("\n[openai] tool sent as:", openai_capture["tools"][0])
print("[openai] system folded into messages[0]:", openai_capture["messages"][0])
print("[openai] parsed LLMResponse:", oresp.text)

## 12. Build your own scenario + agent

The contract for an agent is one method: `respond(history) -> AgentStep`. Combine it with a
`Scenario`, a scripted LLM, and a tool registry, and you can record + replay a brand-new flow with
the same machinery. Here's a minimal echo agent recorded and replayed offline.

In [ ]:
# === OPT-IN: this is the only cell that hits a real provider ===
RUN_LIVE = False                 # <-- set True to actually call the provider (spends credits)
LIVE_PROVIDER = "anthropic"      # anthropic | openai | openrouter | gemini | ollama
LIVE_MODEL = None                # None -> the scenario's model (claude-haiku-4-5); override per provider
LIVE_SCENARIO = "booking_happy"

if not RUN_LIVE:
    print("RUN_LIVE is False — skipping the real call.")
    print("Set RUN_LIVE=True (and pick LIVE_PROVIDER/LIVE_MODEL) to record live.")
else:
    try:
        from dotenv import load_dotenv
        load_dotenv()            # pull keys from .env if present
    except ModuleNotFoundError:
        pass
    from replaygate.capture.providers import make_client

    spec = EXAMPLES[LIVE_SCENARIO]
    inner = make_client(LIVE_PROVIDER, model=LIVE_MODEL)   # the REAL client
    live_dir = WORK / f"live_{LIVE_PROVIDER}_{LIVE_SCENARIO}"

    live_fx = record_conversation(
        agent_factory=spec.build_agent,
        inner_llm=inner,
        scenario=spec.scenario,
        adapter=DirectAdapter(),
        tools=spec.tools(),
        agent_version=f"live-{LIVE_PROVIDER}",
        model=LIVE_MODEL or spec.model,
        recorded_at=datetime.now(timezone.utc),
    )
    write_fixture(str(live_dir), live_fx)
    print(f"recorded LIVE via {LIVE_PROVIDER} → {live_dir}\n")

    for t in live_fx.conversation.turns:
        a = " ".join(m.content for m in t.assistant_messages)
        print(f"  turn {t.index}: {a!r}  tools={[tc.name for tc in t.tool_calls]}")
    # The 'raw' field carries the real provider's response id/model — proof it was a live call:
    for e in live_fx.llm_recording:
        print("  raw:", e["response"].get("raw"))

    # ...and it now replays offline, zero network, byte-faithful:
    replayed = replay_conversation(live_fx, spec.build_agent, spec.tools())
    print("\noffline replay diff:", diff_conversations(live_fx.conversation, replayed))

## Cleanup & next steps

The scratch fixtures live under a temp dir (`WORK`). Run the cell below to remove them.

**Now shipped (this notebook):** replay-and-diff, the cross-turn **invariant suite**
(`replaygate.invariants`), and the **`regress` gate** (`run_regress` + `replaygate regress`) wired
into CI. Sections 6–7 use the real invariant functions; section 8 runs them as a gate.

**Roadmap (per the README):** OpenTelemetry spans wired through the capture loop, channel adapters
beyond `direct` (WhatsApp first), and **cross-version** regress — replay a fixture recorded from one
agent version against a *different* candidate agent, not just the scenario's own agent.

In [ ]:
from replaygate.capture.adapters import Scenario

class EchoAgent:
    # Trivial agent: returns the LLM's text and calls one tool each turn.
    def __init__(self, llm, tools):
        self._llm = llm
        self._tools = tools
        self._n = 0

    def respond(self, history):
        resp = self._llm.create("my-model", "echo system", history, [])
        self._n += 1
        result = self._tools.call("shout", {"text": history[-1]["content"]})
        return AgentStep(assistant_text=resp.text, tool_calls=[ToolCall(
            name="shout", arguments={"text": history[-1]["content"]},
            result=result, call_id=f"e{self._n}")])

def echo_tools():
    return {"shout": lambda text: {"loud": text.upper()}}

class MyScriptedLLM:
    def __init__(self, lines): self._lines = list(lines)
    def create(self, model, system, messages, tools):
        return LLMResponse(text=self._lines.pop(0))

scenario = Scenario(name="echo_demo", user_turns=[["hello"], ["world"]])

custom_fx = record_conversation(
    agent_factory=lambda llm, tools: EchoAgent(llm, tools),
    inner_llm=MyScriptedLLM(["hi there", "still here"]),
    scenario=scenario,
    adapter=DirectAdapter(),
    tools=echo_tools(),
    agent_version="custom-0",
    model="my-model",
    recorded_at=datetime.now(timezone.utc),
)

replayed = replay_conversation(custom_fx, lambda llm, tools: EchoAgent(llm, tools), echo_tools())
print("custom diff:", diff_conversations(custom_fx.conversation, replayed))
for t in replayed.turns:
    print(f"  turn {t.index}: {t.assistant_messages[0].content!r}  "
          f"tool={t.tool_calls[0].name}({t.tool_calls[0].result})")

## Cleanup & next steps

The scratch fixtures live under a temp dir (`WORK`). Run the cell below to remove them.

**Roadmap (per the README):** the cross-turn assertion *suite*, a `regress` command, channel
adapters (WhatsApp first), and a CI gate. The detectors you hand-wrote in sections 6–7 are previews
of that assertion suite.

In [ ]:
import shutil
shutil.rmtree(WORK, ignore_errors=True)
print("removed", WORK)